<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 10 — Simulation Reference

A lookup guide. Notebooks 01 and 02 are meant to be read in order; this one is
meant to be **searched**. Keep it open while you work on Homework 9.

Run the setup cell, then jump to the section you need. Every example makes its
own generator from a seed, so each cell gives the same output every time.

| Section | When you need it |
|---|---|
| [A — Generators and seeds](#A) | making one, passing it on, what a seed does |
| [B — Drawing numbers](#B) | whole numbers, decimals, counts, waiting times |
| [C — Choosing and shuffling](#C) | with and without replacement, random order |
| [D — Python's `random` module](#D) | reading code that uses it; the differences |
| [E — One trial, many trials](#E) | the two layers of every simulation |
| [F — Standard error and intervals](#F) | how sure, and how many trials |
| [G — All the trials at once](#G) | when the loop is too slow |
| [H — Comparing options](#H) | the same random draws for each |
| [I — The bootstrap](#I) | an interval for a statistic of real data |
| [J — Rules that keep results reproducible](#J) | and the bugs they prevent |
| [K — Reporting, and common mistakes](#K) | before anyone else sees it |

The statistics used here (mean, standard deviation, quantiles) and the plots
are in Session 9's reference,
`../09-Visualization-Descriptive-Statistics/03_Statistics_and_Plotting_Reference`.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.4f}".format

---

<a id="A"></a>
## A — Generators and seeds

| You want | Write this |
|---|---|
| a generator with a fixed seed | `rng = np.random.default_rng(8410)` |
| a generator that differs on every run | `rng = np.random.default_rng()` |
| a function that draws random numbers | give it a parameter `rng` and use it |
| the same numbers again | make a new generator **from the same seed** |
| two independent streams from one seed | `first, second = np.random.default_rng(8410).spawn(2)` |

- A **seed** is any whole number. It fixes the entire sequence. There are no
  good or bad seeds; do not pick one because its result looks nice.
- A generator has a **state**: each draw moves it forward. It never goes back
  on its own.
- The same seed gives the same numbers on any machine with the same numpy
  version.

In [ ]:
first = np.random.default_rng(8410)
second = np.random.default_rng(8410)
other = np.random.default_rng(8411)

print("seed 8410:      ", first.integers(0, 100, size=6))
print("seed 8410 again:", second.integers(0, 100, size=6))
print("seed 8411:      ", other.integers(0, 100, size=6))
print("8410, continued:", first.integers(0, 100, size=6))

The pattern for a whole program:

```python
def one_trial(rng):                 # draws from the generator it is given
    return rng.integers(1, 7)


def study(seed, trials):
    rng = np.random.default_rng(seed)        # made once, here
    return [one_trial(rng) for number in range(trials)]
```

---

<a id="B"></a>
## B — Drawing numbers

Every method takes `size=`: leave it out for one number, `size=10` for an
array of ten, `size=(1000, 3)` for a table of 1,000 rows and 3 columns.

| You want | Write this | Gives |
|---|---|---|
| a whole number from `low` to `high − 1` | `rng.integers(low, high)` | `high` is **left out** |
| the same, `high` included | `rng.integers(low, high, endpoint=True)` | |
| a die | `rng.integers(1, 7)` | 1 to 6 |
| a coin | `rng.integers(0, 2)` | 0 or 1 |
| a day of the year | `rng.integers(0, 365)` | 0 to 364 |
| a number from 0 up to (not including) 1 | `rng.random()` | |
| a number between `a` and `b` | `rng.uniform(a, b)` | |
| True with probability `p` | `rng.random() < p` | |
| values clustering around a mean | `rng.normal(mean, standard_deviation)` | can be negative |
| a count of arrivals | `rng.poisson(mean)` | 0, 1, 2, ...; variance equals the mean |
| successes in `n` tries | `rng.binomial(n, p)` | 0 to `n` |
| a waiting time | `rng.exponential(mean)` | positive, skewed |

In [ ]:
rng = np.random.default_rng(1)

print("integers(1, 7, size=8)     ", rng.integers(1, 7, size=8))
print("random(size=3)             ", np.round(rng.random(size=3), 3))
print("uniform(10, 20, size=3)    ", np.round(rng.uniform(10, 20, size=3), 2))
print("normal(170, 8, size=4)     ", np.round(rng.normal(170, 8, size=4), 1))
print("poisson(40, size=8)        ", rng.poisson(40, size=8))
print("binomial(10, 0.3, size=8)  ", rng.binomial(10, 0.3, size=8))
print("exponential(4, size=4)     ", np.round(rng.exponential(4, size=4), 2))
print("integers(1, 7, size=(3, 2))")
print(rng.integers(1, 7, size=(3, 2)))

Which distribution?

| The quantity is | Use |
|---|---|
| one of several equally likely whole numbers | `integers` |
| yes or no with a known chance | `random() < p`, or `binomial(1, p)` |
| how many of `n` independent tries succeed | `binomial` |
| how many independent arrivals in a period | `poisson` |
| the time until the next arrival | `exponential` |
| a measurement with many small influences | `normal` |
| anything in between `a` and `b`, no value preferred | `uniform` |

A quick look at what a distribution gives is worth more than its formula:

In [ ]:
rng = np.random.default_rng(2)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].hist(rng.poisson(40, size=10_000), bins=np.arange(15, 70) - 0.5, color="steelblue")
axes[0].set_title("poisson(40)")
axes[1].hist(rng.normal(40, 6.3, size=10_000), bins=50, color="steelblue")
axes[1].set_title("normal(40, 6.3)")
axes[2].hist(rng.exponential(4, size=10_000), bins=50, color="steelblue")
axes[2].set_title("exponential(4)")
for ax in axes:
    ax.set_xlabel("value")
axes[0].set_ylabel("draws (of 10,000)")
fig.tight_layout()

---

<a id="C"></a>
## C — Choosing and shuffling

| You want | Write this |
|---|---|
| one item | `rng.choice(items)` |
| several, repeats allowed (**with replacement**) | `rng.choice(items, size=5)` |
| several different ones (**without replacement**) | `rng.choice(items, size=5, replace=False)` |
| items with unequal chances | `rng.choice(items, size=5, p=[0.5, 0.3, 0.2])` |
| a random order, as a new array | `rng.permutation(items)` |
| shuffle an array in place | `rng.shuffle(array)` |
| a random sample of rows of a table | `table.sample(n=100, random_state=rng)` |
| a resample of a column, same size | `rng.choice(values, size=len(values), replace=True)` |
| random positions | `rng.integers(0, len(values), size=5)` |

In [ ]:
rng = np.random.default_rng(3)
deck = np.arange(1, 11)

print("with replacement:   ", rng.choice(deck, size=8))
print("without replacement:", rng.choice(deck, size=8, replace=False))
print("a random order:     ", rng.permutation(deck))
print("unequal chances:    ", rng.choice(["rain", "sun", "snow"], size=8, p=[0.3, 0.6, 0.1]))

| | With replacement | Without replacement |
|---|---|---|
| Like | rolling a die, a resample | dealing cards, a survey sample |
| The same item twice | possible | impossible |
| More draws than items | fine | an error |
| `rng.choice` | the default | `replace=False` |

`rng.choice` wants an array or a list of plain values. For a list of
dictionaries or of tuples, choose **positions** and index with them:
`records[rng.integers(0, len(records))]`.

---

<a id="D"></a>
## D — Python's `random` module

Older code, and most tutorials, use the standard library's `random`. You need
to be able to read it. For new code in this course, use a numpy generator.

| `random` | numpy generator | Careful |
|---|---|---|
| `random.seed(1)` | `rng = np.random.default_rng(1)` | `random` has one shared generator |
| `random.randint(1, 6)` | `rng.integers(1, 7)` | `randint` **includes** the upper end |
| `random.randrange(0, 365)` | `rng.integers(0, 365)` | both leave it out |
| `random.random()` | `rng.random()` | |
| `random.uniform(a, b)` | `rng.uniform(a, b)` | |
| `random.gauss(mu, sigma)` | `rng.normal(mu, sigma)` | |
| `random.choice(items)` | `rng.choice(items)` | |
| `random.choices(items, k=5)` | `rng.choice(items, size=5)` | with replacement |
| `random.sample(items, 5)` | `rng.choice(items, size=5, replace=False)` | without |
| `random.shuffle(items)` | `rng.shuffle(items)` | both in place |
| a generator of your own: `random.Random(1)` | `np.random.default_rng(1)` | |

The same seed in the two libraries gives **different** numbers. So does the
old numpy interface (`np.random.seed`, `np.random.randint`, `np.random.rand`),
which is also one shared, global generator.

In [ ]:
random.seed(1)
print("random, seed 1:", [random.randint(1, 6) for roll in range(8)])
print("numpy,  seed 1:", np.random.default_rng(1).integers(1, 7, size=8))

---

<a id="E"></a>
## E — One trial, many trials

Every simulation has these two layers.

```python
def trial(rng):
    """Run the experiment once; return its outcome (a number or True/False)."""
    ...


def run_trials(trial, trials, rng):
    """Call trial(rng) `trials` times; return the outcomes as floats."""
    outcomes = np.empty(trials)
    for number in range(trials):
        outcomes[number] = trial(rng)
    return outcomes
```

| You want | Write this |
|---|---|
| the estimate of a probability | `outcomes.mean()` of True/False outcomes |
| the estimate of an average | `outcomes.mean()` of numeric outcomes |
| a trial with a setting | `lambda rng: trial(rng, setting)`, or a default argument |
| the outcomes as a picture | `ax.hist(outcomes, bins=30)` |
| a trial that returns several numbers | return a tuple; collect into a list, then `pd.DataFrame(rows)` |

In [ ]:
def run_trials(trial, trials, rng):
    outcomes = np.empty(trials)
    for number in range(trials):
        outcomes[number] = trial(rng)
    return outcomes


def any_six(rng, dice=4):
    return (rng.integers(1, 7, size=dice) == 6).any()


rng = np.random.default_rng(4)
print("at least one six in 4 dice:", run_trials(any_six, 10_000, rng).mean(),
      "  exact", round(1 - (5 / 6) ** 4, 4))
print("at least one six in 8 dice:", run_trials(lambda rng: any_six(rng, 8), 10_000, rng).mean(),
      "  exact", round(1 - (5 / 6) ** 8, 4))

Keep the random part and the arithmetic apart whenever you can:

```python
def draw_inputs(rng):          # random, short
    ...
def outcome(inputs, option):   # not random: can be checked by hand
    ...
```

The second function can then be tested on inputs you make up, and several
options can be tried on the same inputs (Section H).

---

<a id="F"></a>
## F — Standard error and intervals

```python
def estimate(outcomes):
    outcomes = np.asarray(outcomes, dtype=float)
    mean = outcomes.mean()
    se = outcomes.std(ddof=1) / np.sqrt(len(outcomes))
    return {"trials": len(outcomes), "mean": mean, "se": se,
            "low": mean - 1.96 * se, "high": mean + 1.96 * se}
```

| You want | Write this |
|---|---|
| the sample standard deviation of an array | `outcomes.std(ddof=1)` (numpy divides by count without `ddof`) |
| the standard error of the mean | `outcomes.std(ddof=1) / np.sqrt(len(outcomes))` |
| a 95 % interval | mean ± 1.96 standard errors |
| a 99 % interval | mean ± 2.58 standard errors |
| the standard error of a share `p`, without the outcomes | `np.sqrt(p * (1 - p) / trials)` |

In [ ]:
def estimate(outcomes):
    outcomes = np.asarray(outcomes, dtype=float)
    mean = outcomes.mean()
    se = outcomes.std(ddof=1) / np.sqrt(len(outcomes))
    return {"trials": len(outcomes), "mean": mean, "se": se,
            "low": mean - 1.96 * se, "high": mean + 1.96 * se}


rng = np.random.default_rng(5)
pd.DataFrame([estimate(run_trials(any_six, count, rng)) for count in (100, 1_000, 10_000)])

### How many trials?

The standard error is `s / sqrt(trials)`, where `s` is the standard deviation
of one outcome. Turn it around:

```text
trials needed = (s / the standard error you want) ** 2
```

Run a small pilot to learn `s`, then compute.

| For a probability near | `s` is about | trials for a standard error of 0.01 | of 0.001 |
|---|---|---|---|
| 0.5 | 0.50 | 2,500 | 250,000 |
| 0.1 or 0.9 | 0.30 | 900 | 90,000 |
| 0.01 or 0.99 | 0.10 | 100 | 10,000 |

- Four times the trials halve the standard error; a hundred times buy one
  decimal place.
- A rare event needs many trials for another reason: with a probability of
  0.001 and 1,000 trials you expect to see it **once**. An estimate built on a
  handful of occurrences is unreliable whatever the formula says.
- An interval is a statement about the method: about 95 of 100 such intervals
  contain the truth, so about 1 in 20 does not.
- Print no more digits than the standard error supports: with a standard error
  of 0.004, write 0.517, not 0.5174.

---

<a id="G"></a>
## G — All the trials at once

Draw a table with one row per trial, and compute along the rows.

| You want | Write this |
|---|---|
| `n` trials of `k` dice | `rolls = rng.integers(1, 7, size=(n, k))` |
| the sum of each trial | `rolls.sum(axis=1)` |
| whether any die in a trial shows 6 | `(rolls == 6).any(axis=1)` |
| whether all do | `(rolls == 6).all(axis=1)` |
| how many do | `(rolls == 6).sum(axis=1)` |
| the largest in each trial | `rolls.max(axis=1)` |
| the smaller of a number and each element | `np.minimum(limit, values)` |
| one value or another, by a condition | `np.where(condition, this, that)` |
| the running total, the running mean | `np.cumsum(x)`, `np.cumsum(x) / np.arange(1, len(x) + 1)` |

`axis=1` means "along each row", giving one result per trial.

In [ ]:
rng = np.random.default_rng(6)
rolls = rng.integers(1, 7, size=(100_000, 4))        # 100,000 trials of four dice

print(rolls[:3])
print("at least one six:", (rolls == 6).any(axis=1).mean())
print("sum of 20 or more:", (rolls.sum(axis=1) >= 20).mean())

- Write the loop version first and check the vectorized one against it. They
  will **not** give the same digits from the same seed, because they draw in a
  different order; they must agree within their standard errors.
- Some simulations cannot be vectorized: when each step depends on the one
  before (a queue, a random walk with a rule, a game with decisions). A loop
  is then the right tool.
- A table of 10 million rows × 50 columns is 4 GB. Run very large simulations
  in batches.

---

<a id="H"></a>
## H — Comparing options

**Draw the random inputs once. Try every option on them.**

```python
demand = rng.poisson(40, size=10_000)                 # drawn once

for option in options:
    result = outcome(option, demand)                  # the same days for all
```

To tell two options apart, estimate their **difference trial by trial**:

```python
gap = outcome(a, demand) - outcome(b, demand)         # one difference per day
estimate(gap)                                         # interval excludes 0?
```

In [ ]:
def profit(baked, demand):
    return 3.0 * np.minimum(baked, demand) - 1.0 * baked


rng = np.random.default_rng(7)
demand = rng.poisson(40, size=10_000)

separately = np.sqrt(estimate(profit(44, demand))["se"] ** 2
                     + estimate(profit(41, rng.poisson(40, size=10_000)))["se"] ** 2)
paired = estimate(profit(44, demand) - profit(41, demand))

print("standard error of the difference, on separate days: {:.3f}".format(separately))
print("standard error of the difference, on the same days:  {:.3f}".format(paired["se"]))
print("44 minus 41: {:+.3f}, interval {:+.3f} to {:+.3f}".format(
    paired["mean"], paired["low"], paired["high"]))

| The interval of the difference | Means |
|---|---|
| lies entirely above 0 | `a` is better; the data tell them apart |
| lies entirely below 0 | `b` is better |
| contains 0 | not distinguishable with this many trials; say so |

"Not distinguishable" is a result. When two options differ by less than
anyone would care about, the honest answer is "either", and the useful
addition is what a wrong choice would cost.

---

<a id="I"></a>
## I — The bootstrap

An interval for **any** statistic of a sample, with no formula: resample the
sample, with replacement, many times.

```python
def bootstrap_interval(values, statistic, resamples, rng):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    results = np.empty(resamples)
    for number in range(resamples):
        resample = rng.choice(values, size=len(values), replace=True)
        results[number] = statistic(resample)
    return tuple(np.quantile(results, [0.025, 0.975]))
```

In [ ]:
def bootstrap_interval(values, statistic, resamples, rng):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    results = np.empty(resamples)
    for number in range(resamples):
        resample = rng.choice(values, size=len(values), replace=True)
        results[number] = statistic(resample)
    return tuple(np.quantile(results, [0.025, 0.975]))


rng = np.random.default_rng(8)
sample = rng.lognormal(mean=1.0, sigma=1.0, size=300)       # a skewed sample

for name, statistic in [("mean", np.mean), ("median", np.median),
                        ("90th percentile", lambda x: np.quantile(x, 0.9))]:
    low, high = bootstrap_interval(sample, statistic, 2000, rng)
    print("{:16} {:6.2f}   interval {:.2f} to {:.2f}".format(name, statistic(sample), low, high))

| Question | Answer |
|---|---|
| How large is a resample? | as large as the sample |
| With replacement? | always; without, every resample is the sample |
| How many resamples? | 1,000 to 10,000; more makes the interval's ends steadier, not narrower |
| What makes the interval narrower? | a larger **sample**, nothing else |
| Which statistics? | anything computed from the sample: mean, median, quantile, correlation, a difference of two groups |
| When does it fail? | tiny samples; the maximum or minimum; data whose rare, huge values the sample happened to miss |
| Two columns (a correlation)? | resample **rows**: draw positions, take both columns at those positions |

```python
positions = rng.integers(0, len(table), size=len(table))
resample = table.iloc[positions]
```

---

<a id="J"></a>
## J — Rules that keep results reproducible

| Rule | The bug it prevents |
|---|---|
| Make **one** generator, at the top, from the seed | results that depend on which function ran first |
| **Pass** it to every function that draws | hidden use of a global generator |
| Never make a generator **inside** a trial or a loop | every trial identical |
| Never call `np.random.seed` or `random.seed` inside a function | restarting the sequence; disturbing other code |
| Do not use `np.random.randint` and the other `np.random.*` functions | one shared, global generator |
| Keep the seed in the output | a result nobody can reproduce |
| Do not try seeds until the result looks good | reporting luck as a finding |

In [ ]:
def trial(rng):
    return rng.integers(1, 7)


right = np.random.default_rng(9)
print("one generator, passed in:     ", [int(trial(right)) for number in range(8)])
print("a generator made on each call:", [int(trial(np.random.default_rng(9))) for number in range(8)])

Checking that a simulation is reproducible takes two lines: run it twice with
the same seed and compare, then once with another seed and see that it
differs.

```python
assert study(seed=1) == study(seed=1)
assert study(seed=1) != study(seed=2)
```

Two **correct** programs can still print different numbers from the same seed,
if they draw in a different order (one die at a time or all at once, demand
before service times or after). That is not a bug. Compare simulations by
whether they agree within their standard errors, never digit for digit.

---

<a id="K"></a>
## K — Reporting, and common mistakes

### What a reported result contains

| Item | Example |
|---|---|
| the estimate, with sensible digits | 0.505 |
| its interval or standard error | 0.495 to 0.515 |
| the number of trials | 10,000 trials |
| the seed | seed 8410 |
| the model | 23 people, 365 equally likely birthdays |
| for a comparison | the difference, its interval, and that both options saw the same draws |

### Symptoms and causes

| Symptom | Likely cause | Fix |
|---|---|---|
| the estimate is exactly 0.0 or 1.0 | a generator is made inside the trial | make it once, pass it in |
| the result changes on every run though a seed is set | some draw uses another generator (`random`, `np.random.*`) | use `rng` everywhere |
| the result never changes when the seed does | the seed is not used, or is overwritten | follow the seed from the top |
| a die never shows 6, or a day 364 never occurs | `rng.integers(1, 6)`: the upper end is left out | `rng.integers(1, 7)` |
| a die shows 7 | code moved from `random.randint(1, 6)` to `rng.integers(1, 7, endpoint=True)` or similar | check the ends |
| the standard error does not shrink with more trials | it is the standard deviation: the `/ sqrt(n)` is missing | divide |
| the interval is a little too narrow | `std()` without `ddof=1` | `std(ddof=1)` |
| two options seem to swap places from run to run | each is tried on its own random draws | common random numbers (Section H) |
| `ValueError: Cannot take a larger sample than population when replace is False` | more draws than items, without replacement | `replace=True`, or fewer draws |
| `ValueError: a must be 1-dimensional` | `rng.choice` on a list of tuples or dicts | choose positions with `rng.integers` |
| `nan` for the standard error | fewer than two outcomes | more trials |
| the bootstrap interval has zero width | resampling without replacement | `replace=True` |
| the simulation takes minutes | a Python loop over millions of trials | vectorize (Section G), or fewer trials if the interval is already narrow enough |
| a result looked better with a different seed | that is luck | report the first seed; show several |